### 目录

1、数据库概览方式

2、数据清洗的主要目标及流程

3、数据清洗方式

3.1 处理不一致值

3.2 处理异常值

3.3 处理缺失值

3.4 处理不合理值及重复值

4、数据类别

5、特征构造与编码

### 1、数据库概览方式

数据集形状、前五行、基本信息、统计描述

检查空值、列名、列统计

In [1]:
import pandas as pd
import numpy as np

# 严格按照菜鸟教程要求，构造【含5类数据问题】的原始脏数据
df = pd.DataFrame({
    # 用户ID（正常主键）
    "用户ID": [1001, 1002, 1003, 1004, 1002, 1005, 1006, 1007, 1008, 1009],
    # 姓名：含缺失值
    "姓名": ["张三", "李四", "王五", np.nan, "李四", "钱七", "孙八", "周九", "吴十", np.nan],
    # 性别：不一致值（male/M/女/F/female）+ 混合格式
    "性别": ["男", "male", "M", "女", "male", "F", "female", "男", "F", "女"],
    # 年龄：缺失值 + 错误值(负数) + 异常值(150)
    "年龄": [25, 30, -6, 45, 30, 150, 28, np.nan, 32, 40],
    # 城市：不一致值(大小写混合) + 缺失值
    "城市": ["beijing", "Shanghai", "guangzhou", "shenzhen", "Shanghai", np.nan, "Beijing", "hangzhou", "Nanjing", "shenzhen"],
    # 年收入：异常值(极端大) + 错误值(负数)
    "年收入": [80000, 120000, 75000, 95000, 120000, -5000, 110000, 98000, 300000, 105000],
    # 上次消费金额：缺失值
    "上次消费金额": [200, 500, np.nan, 300, 500, 150, np.nan, 400, 250, 350]
})

# 导出为CSV文件（和.ipynb同目录，无索引，纯原始脏数据）
df.to_csv("customer_data.csv", index=False, encoding="utf-8-sig")

print("数据集形状（行，列）:", df.shape)
print("\n数据前5行：", df.head())
print("\n数据基本信息：")
df.info()
print("\n数据统计描述：")
print(df.describe())
print("\n每列空值数量")
print(df.isnull().sum())
print("\n数据集所有列名：", df.columns.tolist())
print("\n姓名列值统计：")
print(df["姓名"].value_counts())


### 2、数据清洗的主要目标

流程：不一致值、异常值、缺失值、不合理值、重复值

In [1]:
import pandas as pd

# 构建数据质量维度数据表
data_quality = pd.DataFrame({
    "目标(英文)": ["完整性(Completeness)", "一致性(Consistency)", "准确性(Accuracy)", "唯一性(Uniqueness)", "合理性(Validity)"],
    "对象": ["缺失值", "不一致值", "异常值", "重复值", "错误值"],
    "描述": [
        "数据记录中某些字段的值为空（NaN, NULL）",
        "数据格式、单位或编码不统一（如\"男\"、\"Male\"、\"M\"）",
        "与大多数数据明显偏离的极端值，扭曲统计结果，识别（IQR、Z-Score）后删除或修正",
        "数据集中存在完全相同的记录，使模型过度偏向重复样本，影响泛化能力",
        "确保数据符合逻辑或业务规则（如年龄不能为负）"
    ]
})

data_quality

### 3、数据清洗方式

#### 3.1 不一致值：映射统一编码+统一数据类型

In [ ]:
# 重新加载原始数据
df = pd.read_csv('customer_data.csv')

# 步骤1：去除字符串前后空格
df['性别'] = df['性别'].str.strip()

# 步骤2：映射统一编码（核心方法）
gender_map = {'男': '男', 'Male': '男', 'M': '男', '女': '女', 'F': '女'}
df['性别'] = df['性别'].replace(gender_map)

# 步骤3：城市统一小写
df['城市'] = df['城市'].str.lower()

# 步骤4：转换数据类型：确保数值列是数字类型
if '年龄' in df.columns:
    df['年龄'] = pd.to_numeric(df['年龄'], errors='coerce')

print("处理不一致值后的数据：")
print(df.head())

#### 3.2 处理异常值

异常值是指与大多数数据明显偏离的极端值，会扭曲统计结果。
常用识别方法：
- **IQR四分位距法**：Q1-1.5*IQR 以下 或 Q3+1.5*IQR 以上为异常值
- **Z-Score法**：Z-score > 3 或 < -3 的值为异常值

In [ ]:
import pandas as pd
import numpy as np

# 重新加载原始数据
df = pd.read_csv('customer_data.csv')

# IQR法检测异常值
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# 检测年龄异常值
age_outliers, age_lower, age_upper = detect_outliers_iqr(df, '年龄')
print(f"年龄异常值检测 - 下界: {age_lower}, 上界: {age_upper}")
print(f"年龄异常值: {age_outliers['年龄'].tolist()}")

# 检测年收入异常值
income_outliers, income_lower, income_upper = detect_outliers_iqr(df, '年收入')
print(f"\n年收入异常值检测 - 下界: {income_lower}, 上界: {income_upper}")
print(f"年收入异常值: {income_outliers['年收入'].tolist()}")

##### 异常值处理方法

In [ ]:
import pandas as pd
import numpy as np

# 重新加载原始数据
df = pd.read_csv('customer_data.csv')

# 方法1：用边界值替换异常值
def cap_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    data[column] = data[column].clip(lower=lower_bound, upper=upper_bound)
    return data

# 对年龄进行异常值处理
df = cap_outliers(df, '年龄')

# 对年收入进行异常值处理
df = cap_outliers(df, '年收入')

print("异常值处理（用边界值替换）后的数据：")
print(df)

#### 3.3 处理缺失值

缺失数据的类型

（1）随机丢失（MAR，Missing at Random）

随机丢失意味着数据丢失的概率与丢失的数据本身无关，而仅与部分已观测到的数据有关。
也就是说，数据的缺失不是完全随机的，该类数据的缺失依赖于其他完全变量。

（2）完全随机丢失（MCAR，Missing Completely at Random）

数据的缺失是完全随机的，不依赖于任何不完全变量或完全变量，**不影响样本的无偏性**。
简单来说，就是数据丢失的概率与其假设值以及其他变量值都完全无关。

（3）非随机丢失（MNAR，Missing not at Random）

数据的缺失与不完全变量自身的取值有关。
分为两种情况：**缺失值取决于其假设值**（例如，高收入人群通常不希望在调查中透露他们的收入）；
或者，**缺失值取决于其他变量值**（假设女性通常不想透露她们的年龄，则这里年龄变量缺失值受性别变量的影响）。

![数据清洗流程图](1.jpg)

##### 3.3.1 删除法

In [ ]:
import pandas as pd

# 重新加载原始数据
df = pd.read_csv('customer_data.csv')

# 删除任何包含缺失值的行（适用于缺失值很少的情况）
df_dropped = df.dropna()
print("删除缺失值后的数据形状：", df_dropped.shape)
print(df_dropped)

##### 3.3.2 填充/插补法

###### 平均值填充（Mean/Mode Completer）
------------------------------------------------------------------
将初始数据集中的属性分为数值属性和非数值属性来分别进行处理。

如果空值是**数值型**的，就根据该属性在其他所有对象的取值的**平均值**来填充该缺失的属性值； 如果空值是**非数值型**的，就根据统计学中的**众数**原理，用该属性在其他所有对象的取值次数最多的值(即出现频率最高的值)来补齐该缺失的属性值。

与其相似的另一种方法叫**条件平均值填充法**（Conditional Mean Completer）。在该方法中，用于求平均的值并不是从数据集的所有对象中取，而是从与该对象具有**相同决策属性值**的对象中取得。

In [ ]:
import pandas as pd

# 重新加载原始数据
df = pd.read_csv('customer_data.csv')

# 单一值填充
# 对于数值型列（如'年龄'），用中位数填充（比均值更抗异常值影响）
if '年龄' in df.columns:
    df['年龄'].fillna(df['年龄'].median(), inplace=True)

# 对于分类列（如'城市'），用众数（最频繁出现的值）填充
if '城市' in df.columns:
    df['城市'].fillna(df['城市'].mode()[0], inplace=True)

# 姓名列用众数填充
df['姓名'].fillna(df['姓名'].mode()[0], inplace=True)

# 上次消费金额用中位数填充
df['上次消费金额'].fillna(df['上次消费金额'].median(), inplace=True)

print("中位数/众数填充后的数据：")
print(df)

###### 更多高级填充方法

以下四种高级填充方法已移至单独的文件 [imputation_methods.ipynb](imputation_methods.ipynb)：

1. **条件平均值填充** - 按分组进行均值填充
2. **热卡填充（Hot deck imputation）** - 基于相似样本填充
3. **KNN填充** - K近邻算法填充
4. **回归填充（Regression）** - 建立回归模型预测填充

请查看 `imputation_methods.ipynb` 文件获取详细代码和示例。

```python
# 高级填充方法代码示例结构：

# 1. 条件平均值填充
# df["年龄"] = df.groupby("性别")["年龄"].transform(lambda x: x.fillna(x.mean()))

# 2. 热卡填充 - 基于相似样本的距离匹配
# 使用 pairwise_distances 计算相似度

# 3. KNN填充 - 使用 sklearn.impute.KNNImputer

# 4. 回归填充 - 使用 LinearRegression 预测缺失值
```

In [ ]:
# 高级填充方法代码已移至 imputation_methods.ipynb
# 请查看该文件获取完整的条件平均值填充、热卡填充、KNN填充、回归填充代码
print("请查看 imputation_methods.ipynb 获取高级填充方法详细代码")

#### 3.4 处理不合理值及重复值

##### 3.4.1 处理不合理值
确保数据符合逻辑或业务规则（如年龄不能为负数）

In [ ]:
import pandas as pd
import numpy as np

# 重新加载原始数据
df = pd.read_csv('customer_data.csv')

# 处理负数年龄（设为NaN后用均值填充）
df.loc[df['年龄'] < 0, '年龄'] = np.nan
df['年龄'].fillna(df['年龄'].mean(), inplace=True)

# 处理负数年收入（设为NaN后用均值填充）
df.loc[df['年收入'] < 0, '年收入'] = np.nan
df['年收入'].fillna(df['年收入'].mean(), inplace=True)

print("不合理值处理后的数据：")
print(df)

##### 3.4.2 处理重复值

In [ ]:
import pandas as pd

# 重新加载原始数据
df = pd.read_csv('customer_data.csv')

# 检测重复行
duplicates = df[df.duplicated()]
print("重复行：")
print(duplicates)

# 删除重复行（保留第一次出现的记录）
df_no_dup = df.drop_duplicates()
print(f"\n删除重复行后的数据形状： {df_no_dup.shape}")
print(df_no_dup)

### 4、数据类别

数据的几种基本类型：
- **数值型数据**：连续型（如年龄、收入）、离散型（如数量）
- **分类型数据**：有序型（如学历）、无序型（如性别、颜色）
- **时间序列数据**：带有时间戳的数据
- **文本数据**：字符串形式的自然语言

In [ ]:
import pandas as pd

# 重新加载原始数据
df = pd.read_csv('customer_data.csv')

# 查看数据类型
print("各列数据类型：")
print(df.dtypes)

# 数值型列
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"\n数值型列: {numeric_cols}")

# 分类型列
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"分类型列: {categorical_cols}")

### 5、特征构造与编码

##### 5.1 特征构造
根据已有特征创造新的有意义的特征

In [ ]:
import pandas as pd

# 重新加载原始数据
df = pd.read_csv('customer_data.csv')

# 特征构造示例
# 1. 构造消费能力指标（上次消费金额 / 年收入 * 10000）
df['消费能力指数'] = df['上次消费金额'] / df['年收入'] * 10000

# 2. 构造年龄段
df['年龄段'] = pd.cut(df['年龄'], bins=[0, 25, 35, 50, 100], labels=['青年', '中年', '壮年', '老年'])

print("特征构造后的数据：")
print(df)

##### 5.2 特征编码

将分类型数据转换为数值型，以便机器学习算法使用

常用方法：
- **标签编码（Label Encoding）**：将类别映射为0, 1, 2, ...
- **独热编码（One-Hot Encoding）**：将每个类别转换为独立的二进制列

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# 重新加载原始数据
df = pd.read_csv('customer_data.csv')

# 方法1：标签编码
le = LabelEncoder()
df['性别_标签编码'] = le.fit_transform(df['性别'])
print("标签编码结果：")
print(df[['性别', '性别_标签编码']])

# 方法2：独热编码
df_city = pd.get_dummies(df['城市'], prefix='城市')
df = pd.concat([df, df_city], axis=1)
print("\n独热编码结果：")
print(df.head())